# LLM Maths Demo

Small LLMs are notoriously bad with maths (which is funny because one forward pass through a transfomer model has many many matrix multiplications and additions!). We can use our `SymbolicModel` to probe what functions a small LLM is actually using when carrying out mathematical operations.

In this demo, we use the small model Llama-3.2-1B-Instruct. Depending on your laptop, you should be able to run this whole notebook locally!

## Set-up

In [37]:
import numpy as np
from symtorch import SymbolicModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re

In [60]:
# This is the model we are going to use
model_name = "meta-llama/Llama-3.2-1B-Instruct"

# Load the tokenizer and the model
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

torch.manual_seed(290402)
# For our experiment, we want a deterministic model
torch.use_deterministic_algorithms(True)

# Function which calls our LLM
def llm_call(prompt: str) -> str:
    inputs = tok(prompt, return_tensors="pt")

    out = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False,          # greedy
    )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tok.decode(new_tokens, skip_special_tokens=True).strip()

Let's try out our LLM to see how it performs at basic addition.

In [ ]:
output = llm_call("Return only the numeric answer in the format $boxed$. What is 12+7=?")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [62]:
print(output)

.

## Step 1: We need to add 12 and 7 together.
## Step 2: The result of the addition is 19.
## Step 3: We need to put the result in the format $boxed$.
## Step 4: The final answer is $\boxed{19}$.

The final answer is: $\boxed{19}$


For smaller numbers it can perform reasonably well. Let's see it's behvaiour for larger (3 digit) numbers.

In [84]:
output = llm_call("Return only the numeric answer in the format $boxed$. What is 972+373=?")
print(output)

print("True answer = ", 972+373)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


## Step 1: Add the two numbers together
First, we need to add 972 and 373 together.

## Step 2: Calculate the sum
972 + 373 = 1445

## Step 3: Format the answer
The answer should be in the format $boxed{1445}$.

The final answer is: $\boxed{1445}$
True answer =  1345


No longer performs that great! 

We can use `SymbolicModel` to approximate the functions that the LLM is using when performing maths.

In [85]:
# Function to extract the number that the LLM returns
def extract_boxed_number(text: str) -> float:
    # Try $\boxed{...}$ format first
    match = re.search(r'\$\\boxed\{([^}]+)\}\$', text)
    if match:
        return float(match.group(1))
    # Try $boxed{...}$ format (without backslash)
    match = re.search(r'\$boxed\{([^}]+)\}\$', text)
    if match:
        return float(match.group(1))
    # Try \boxed{...} without dollar signs
    match = re.search(r'\\boxed\{([^}]+)\}', text)
    if match:
        return float(match.group(1))
    # Fallback: try to find any number after an equals sign
    match = re.search(r'=\s*(\d+)', text)
    if match:
        return float(match.group(1))
    raise ValueError(f"No boxed number found in: {text}")

# Function to create a dataset of random number pairs 
def random_number_pairs(N = 100, maximum = 999):
    return np.random.randint(0, maximum, size=(N, 2))

## Addition

`SymbolicModel` is model-agnostic. You just need to pass a function that is of the form `f(inputs) = outputs`.

In [96]:
# Create a function that the SymbolicModel expects 
def llm_addition(X):
    outputs = []
    # X is of shape (N,2)
    for n in range(X.shape[0]):
        a = X[n,0]
        b = X[n,1]
        output = llm_call(f"Return only the numeric answer in the format $boxed$. What is {int(a)}+{int(b)}=?")
        output = extract_boxed_number(output)
        outputs.append(output)
    return np.array(outputs)

Create a random dataset of numbers to add.

In [97]:
np.random.seed(290402)

X = random_number_pairs(50)

Example of the numbers in our dataset.

In [98]:
print(X[:5,:])

[[451  41]
 [871 582]
 [237 193]
 [661 992]
 [417 724]]


In [99]:
# Initialise our model
symbolic_model_addition = SymbolicModel(sr_params={'constraints': {'sin':1, 'exp':1}})

In [100]:
#Perform SR on our model
symbolic_model_addition.fit(llm_addition, X)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

Running SR on output dimension 0 of 0


/Users/liz/PhD/SymTorch_project/symtorch_venv/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 2.260e+06
Progress: 11317 / 12400 total iterations (91.266%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.522e+05  0.000e+00  y = 1002.9
3           8.858e+03  1.414e+00  y = x₁ + x₀
5           8.173e+03  4.001e-02  y = x₀ + (x₁ * 0.95781)
7           7.936e+03  1.442e-02  y = (x₀ * 1.0399) + (x₁ * 0.93206)
9           7.932e+03  1.580e-05  y = ((x₀ + -5.5577) * 1.0447) + (x₁ * 0.93728)
10          1.657e+03  1.566e+00  y = (x₀ + x₁) + inv((x₀ * -0.016272) + 1.1211)
11          1.633e+03  1.412e-02  y = (x₀ + x₁) + inv(sin(x₀) + 0.11312)
12          1.491e+03  9.088e-02  y = x₀ + ((x₁ * 0.97844) + inv((x₀ * -0.016272) + 1.1211))
13          1.435e+03  3.812e-02  y = (x₀ + (x₁ * 0.97747)) + inv(sin(x₀) + 0.11302)
15          1.416e+03  6.665e

[ Info: Final population:
[ Info: Results saved to:


  - SR_output/SymbolicModel/dim0_1763721736/hall_of_fame.csv


In [101]:
symbolic_model_addition.get_equation()

'(x0 + ((inv(x0 + -938.28375) * x1) + x1)) + (inv(sin(x0) + 0.1123411) * 1.4623601)'